In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_events_daily AS
WITH source_data AS (
  SELECT
    md5(concat_ws('||', 
      coalesce(cast(employee_key as string), ''), 
      coalesce(trim(employee_full_name), ''), 
      coalesce(trim(event_type), ''), 
      coalesce(trim(swap_cell), ''), 
      coalesce(cast(start_datetime as string), ''), 
      coalesce(cast(end_datetime as string), ''), 
      coalesce(_source_file, '')
    )) AS event_key,
    
    CAST(employee_key AS INT) AS employee_key,
    trim(employee_full_name) AS employee_full_name,
    trim(event_type) AS event_type,
    
    -- Standaryzacja stacji docelowej
    CASE
      WHEN swap_cell IS NOT NULL AND trim(swap_cell) != '' THEN upper(trim(swap_cell))
      WHEN (swap_cell IS NULL OR trim(swap_cell) = '') AND remarks IS NOT NULL AND trim(remarks) != '' THEN upper(trim(remarks))
      ELSE 'NO ASSIGNMENT'
    END AS cell_code,

    -- Konwersja dat i godzin bazowych
    CAST(CAST(start_datetime AS TIMESTAMP) AS DATE) AS start_date,
    date_format(CAST(start_datetime AS TIMESTAMP), 'HH:mm:ss') AS start_time_str,
    CAST(start_datetime AS TIMESTAMP) AS start_datetime,
    
    CAST(CAST(end_datetime AS TIMESTAMP) AS DATE) AS end_date,
    date_format(CAST(end_datetime AS TIMESTAMP), 'HH:mm:ss') AS end_time_str,
    CAST(end_datetime AS TIMESTAMP) AS end_datetime,
    
    CAST(fte AS DOUBLE) AS fte,
    trim(remarks) AS remarks,
    source,
    _source_file,
    _bronze_ingested_at
  FROM data_warehouse_factory.bronze.bronze_events
  WHERE start_datetime IS NOT NULL AND end_datetime IS NOT NULL
),

-- 1. Rozbicie zakresu dat na poszczególne dni kalendarzowe
expanded_days AS (
  SELECT 
    s.*,
    explode(sequence(s.start_date, s.end_date, interval 1 day)) AS activity_date
  FROM source_data s
),

-- 2. Logika wyznaczania godzin granicznych dla każdego dnia produkcyjnego
calculated_hours AS (
  SELECT 
    event_key,
    employee_key,
    employee_full_name,
    event_type,
    cell_code,
    
    start_date AS start_date_general,
    end_date AS end_date_general,
    activity_date AS production_date,
    
    CASE 
      WHEN activity_date = start_date THEN start_time_str
      ELSE '06:00:00'
    END AS daily_start_time_str,

    CASE 
      WHEN activity_date = end_date THEN end_time_str
      ELSE '14:00:00'
    END AS daily_end_time_str,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at
  FROM expanded_days
),

-- 3. Zbudowanie timestampów i klucza daty
building_timestamps AS (
  SELECT
    event_key,
    employee_key,
    employee_full_name,
    event_type,
    cell_code,
    start_date_general,
    end_date_general,
    production_date,
    CAST(date_format(production_date, 'yyyyMMdd') AS INT) AS date_key,
    
    to_timestamp(concat(cast(production_date as string), ' ', daily_start_time_str)) AS daily_event_start,
    to_timestamp(concat(cast(production_date as string), ' ', daily_end_time_str)) AS daily_event_end,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at
  FROM calculated_hours
),

-- 4. Wyliczenie czasów w interwałach
intervals_summary AS (
  SELECT 
    md5(concat_ws('||', event_key, cast(production_date as string))) AS daily_event_key,
    event_key AS parent_event_key,
    date_key,
    employee_key,
    employee_full_name,
    event_type,
    cell_code,
    
    start_date_general,
    end_date_general,
    production_date,
    
    date_format(daily_event_start, 'HH:mm:ss') AS start_time,
    date_format(daily_event_end, 'HH:mm:ss') AS end_time,
    daily_event_start,
    daily_event_end,
    
    ROUND(timestampdiff(MINUTE, daily_event_start, daily_event_end) / 60.0, 2) AS duration_hours,
    ROUND(timestampdiff(MINUTE, daily_event_start, daily_event_end), 2) AS duration_minutes,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at,
    current_timestamp() AS _silver_ingested_at
  FROM building_timestamps
  WHERE daily_event_start < daily_event_end
)

-- 5. Dołączenie klucza komórki bezpośrednio w Silver
SELECT 
  s.daily_event_key,
  s.parent_event_key,
  s.date_key,
  s.employee_key,
  c.cell_key,
  s.cell_code,
  s.employee_full_name,
  s.event_type,
  s.start_date_general,
  s.end_date_general,
  s.production_date,
  s.start_time,
  s.end_time,
  s.daily_event_start,
  s.daily_event_end,
  s.duration_hours,
  s.duration_minutes,
  s.fte,
  s.remarks,
  s.source,
  s._source_file,
  s._bronze_ingested_at,
  s._silver_ingested_at
FROM intervals_summary s
LEFT JOIN data_warehouse_factory.silver.silver_production_structure c
  ON s.cell_code = c.cell_code;